# 03 — Trends and Rankings

Time-series analysis: how the U.S. ranks among its 6 peers over the decade, rate of change comparisons, and linear trend slopes.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
from scipy import stats as sp_stats
from helpers import (
    load_data, get_country, get_countries, get_peer_average,
    PEER_COUNTRIES, PEER_COUNTRIES_NO_US,
    FOCUS_VARIABLES, FOCUS_COLS, VAR_LABELS, VAR_LOWER_IS_BETTER,
    COUNTRY_NAMES, COLORS
)

df = load_data()
peers_df = get_countries(df, PEER_COUNTRIES)
usa = get_country(df, 'USA')
YEARS = sorted(df['Year'].unique())

## 1. U.S. Rank Among 6 Peers — Every Year

In [ ]:
all_ranks = []

for year in YEARS:
    year_data = peers_df[peers_df['Year'] == year]
    
    for col, label, goal, lower_is_better in FOCUS_VARIABLES:
        subset = year_data[['Country.Code', col]].dropna()
        if len(subset) < 3:
            continue
        
        if lower_is_better:
            subset['Rank'] = subset[col].rank(ascending=True)
        else:
            subset['Rank'] = subset[col].rank(ascending=False)
        
        for _, row in subset.iterrows():
            all_ranks.append({
                'Year': year,
                'Country': row['Country.Code'],
                'Country Name': COUNTRY_NAMES.get(row['Country.Code'], row['Country.Code']),
                'Indicator': label,
                'Column': col,
                'Value': row[col],
                'Rank': int(row['Rank']),
            })

ranks_df = pd.DataFrame(all_ranks)

# Show USA ranks summary
usa_ranks = ranks_df[ranks_df['Country'] == 'USA'].pivot(
    index='Indicator', columns='Year', values='Rank'
)
usa_ranks['Avg Rank'] = usa_ranks.mean(axis=1).round(1)
usa_ranks.sort_values('Avg Rank', ascending=False)

## 2. Trend Slopes — U.S. vs. Peer Average

Fit linear trends (OLS) to measure rate of change over the decade.

In [ ]:
peer_avg = get_peer_average(df)

slope_results = []

for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    # USA trend
    usa_series = usa[['Year', col]].dropna()
    if len(usa_series) < 5:
        continue
    
    usa_slope, usa_intercept, usa_r, usa_p, usa_se = sp_stats.linregress(
        usa_series['Year'], usa_series[col]
    )
    
    # Peer average trend
    peer_series = peer_avg[col].dropna().reset_index()
    if len(peer_series) < 5:
        continue
    
    peer_slope, peer_intercept, peer_r, peer_p, peer_se = sp_stats.linregress(
        peer_series['Year'], peer_series[col]
    )
    
    # Percent change 2006-2015
    usa_2006 = usa_series[usa_series['Year'] == 2006][col]
    usa_2015 = usa_series[usa_series['Year'] == 2015][col]
    
    usa_pct = None
    if len(usa_2006) > 0 and len(usa_2015) > 0:
        v06, v15 = usa_2006.iloc[0], usa_2015.iloc[0]
        if v06 != 0:
            usa_pct = ((v15 - v06) / abs(v06)) * 100
    
    slope_results.append({
        'Indicator': label,
        'USA Slope (per yr)': round(usa_slope, 3),
        'USA R²': round(usa_r**2, 3),
        'USA p-value': round(usa_p, 4),
        'Peer Slope (per yr)': round(peer_slope, 3),
        'Peer R²': round(peer_r**2, 3),
        'USA % Change': round(usa_pct, 1) if usa_pct else None,
        'Converging?': 'Yes' if (lower_is_better and usa_slope < peer_slope) or 
                                 (not lower_is_better and usa_slope > peer_slope) else 'No',
    })

slopes_df = pd.DataFrame(slope_results)
slopes_df.sort_values('USA % Change', key=lambda x: x.abs(), ascending=False)

## 3. U.S. vs. Peer Average — Time Series Values

In [ ]:
# Build a tidy dataframe for time-series plotting
ts_data = []

for year in YEARS:
    usa_year = usa[usa['Year'] == year]
    peer_year = df[
        (df['Country.Code'].isin(PEER_COUNTRIES_NO_US)) &
        (df['Year'] == year)
    ]
    
    for col, label, goal, lib in FOCUS_VARIABLES:
        # USA value
        if len(usa_year) > 0:
            val = usa_year.iloc[0][col]
            if pd.notna(val):
                ts_data.append({
                    'Year': year, 'Entity': 'United States',
                    'Indicator': label, 'Column': col, 'Value': val
                })
        
        # Peer average
        peer_mean = peer_year[col].mean()
        if pd.notna(peer_mean):
            ts_data.append({
                'Year': year, 'Entity': 'Peer Average',
                'Indicator': label, 'Column': col, 'Value': peer_mean
            })
        
        # Individual peers
        for _, prow in peer_year.iterrows():
            pval = prow[col]
            if pd.notna(pval):
                ts_data.append({
                    'Year': year,
                    'Entity': COUNTRY_NAMES.get(prow['Country.Code'], prow['Country.Code']),
                    'Indicator': label, 'Column': col, 'Value': pval
                })

ts_df = pd.DataFrame(ts_data)
print(f"Time-series dataset: {ts_df.shape}")
ts_df.head(10)

## 4. Save Processed Data

In [ ]:
ranks_df.to_csv('../analysis/yearly_ranks.csv', index=False)
slopes_df.to_csv('../analysis/trend_slopes.csv', index=False)
ts_df.to_csv('../analysis/timeseries.csv', index=False)

print('Saved: yearly_ranks.csv, trend_slopes.csv, timeseries.csv')